# Merge TCIA metadata with mask statistics
This notebook is used to create the final metadata.csv. It combines mask statistics (number of components, skeleton size per component, and number of voxels per component) with the metadata from TCIA (Instance UID, Patient Age, Patient Sex, Manufacturer, Manufacturer Model Name, Software Versions).

In [69]:
import pandas as pd
import re

In [70]:
# load data
metadata = pd.read_csv("../../../data/raw/metadata/merged_data_all.csv")
final_dataset = pd.read_csv("../../../data/processed/final_dataset/final_dataset_statistics.csv")

# our annotations
post_processing_status = pd.read_csv("../../../data/processed/manual_annotation/final_dataset_postprocessing/manual_postprocessing_final_dataset.csv")

# filenames form HQColon
filenames_hqcolon = pd.read_csv("../../../data/raw/metadata/filenames_non_collapsed_renamed.txt", header=None, names=["filename"])

In [71]:
# extract old filenames
def extract_sub_id(filename):
    match_1 = re.match(r"colon_(0*\d+)-", filename)
    match_2 = re.match(r"^colon_\d+-(prone|supine).*\.mha$", filename)

    if match_1 and match_2:
        # sub001_pos-prone_scan-1
        return f"sub{int(match_1.group(1)):03d}_pos-{match_2.group(1)}_scan-1"
    return None

final_dataset["name"] = final_dataset["filename"].apply(extract_sub_id)

# merge with post_processing_status (accepted, accepted with edits, rejected). However, can not be rejected, as it represents the final dataset.
final_dataset_merged = pd.merge(
    final_dataset,
    post_processing_status[["filename", "status"]],
    on="filename",
    how="left"
)

# merge with TCIA metadata
final_dataset_merged = pd.merge(
    final_dataset_merged,
    metadata[["Patient ID", "Patient Sex", "Patient Age", "Position", "name", "Series Instance UID", "Manufacturer", "Manufacturer Model Name", "Software Versions"]],
    on="name",
    how="left"
)

# rename column header
final_dataset_merged = final_dataset_merged.rename(columns={"Patient ID": "patient_id", "Patient Sex": "patient_sex", "Patient Age": "patient_age", "Position": "patient_position", "Series Instance UID": "instance_uid", "Manufacturer": "manufacturer", "Manufacturer Model Name": "manufacturer_model_name", "Software Versions": "software_versions"})
final_dataset_merged = final_dataset_merged.drop(columns="name")

In [ ]:
# check for basic merging errors
print(f"Count unique instance_uid (should be 1032): {len(final_dataset_merged["instance_uid"].unique())}")
print(f"Patient positions missing (should be 0): {len(final_dataset_merged[final_dataset_merged["patient_position"].isna()]["patient_position"].unique())}")

Count unique instance_uid (should be 1032): 1032
Patient positions missing (should be 0): 0


In [72]:
final_dataset_merged.to_csv("../../../data/processed/final_dataset/final_dataset_statistics_and_metadata.csv")

### Count # of accepted, accepted with edits, and rejected masks
We ran inference on most raw scans. This means that we have to filter out the ones that are already part of $D_{HQColon}$, as we focusing on additional masks.

The final dataset already consists only of additional masks. Therefore, no additional filtering is required. For the rejected ones, we don't count the cases that are part of $D_{HQColon}$. This includes cases where $D_{HQColon}$ used a different scan for the same patient position configuration.

In [68]:
print(final_dataset_merged[["filename", "status"]].drop_duplicates()["status"].value_counts())
rejected = post_processing_status[post_processing_status["status"] == "reject"]["filename"].to_list()
filenames_hqcolon_list = filenames_hqcolon["filename"].to_list()

# filter out duplicates
duplicates = []
for file in rejected:
    if file in filenames_hqcolon_list:
        duplicates.append(file)
        
print(f"Rejected: {len(rejected)-len(duplicates)}")

status
accept               934
accept with edits     98
Name: count, dtype: int64
Rejected: 182
